## Parsing Playground

we can try some tools to parse pdf with a lot of images.
- Using Docling local vision model
- Using Vision model API services (e.g. Gemini vision, LLamaIndex)

Using local model is free, but limited to model size

Using on premise API service needs additional cost to implement, but provides better model to choose without hardware limitations

### Local

In [1]:
# install dependencies
%pip install "docling[vlm]" "transformers" "torch" "torchvision"

  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached pluggy-1.6.0-py3-none-any.whl.metadata (4.8 kB)
  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached scipy-1.15.3-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached mdurl-0.1.2-py3-none-any.whl.met

In [1]:
import os

os.environ["TORCH_COMPILE_DISABLE"] = "1"
from pypdf import PdfReader, PdfWriter
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions, 
    smolvlm_picture_description
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc import PictureItem

def prepare_pdf_subset(input_pdf_path, output_pdf_path, num_pages):
    """
    Fungsi untuk mengambil n-halaman pertama dari PDF.
    Ini sangat penting untuk menghemat VRAM dan waktu proses saat testing.
    """
    reader = PdfReader(input_pdf_path)
    writer = PdfWriter()
    
    # Ambil halaman sesuai jumlah yang diminta (atau maksimal halaman yang ada)
    total_pages = len(reader.pages)
    pages_to_extract = min(num_pages, total_pages)
    
    for i in range(pages_to_extract):
        writer.add_page(reader.pages[i])
        
    with open(output_pdf_path, "wb") as f_out:
        writer.write(f_out)
        
    print(f"PDF berhasil dipotong menjadi {pages_to_extract} halaman pertama.")
    return output_pdf_path

def parse_pdf_to_unified_text(pdf_path, max_pages=5):
    # 1. Siapkan PDF sementara dengan jumlah halaman terbatas
    temp_pdf = "temp_subset.pdf"
    prepare_pdf_subset(pdf_path, temp_pdf, max_pages)
    
    # 2. Konfigurasi Pipeline Docling dengan SmolVLM
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_picture_description = True  
    pipeline_options.picture_description_options = smolvlm_picture_description
    
    # Prompt untuk fokus pada UI
    pipeline_options.picture_description_options.prompt = (
        "This is a screenshot of a web application interface. "
        "Describe the visible UI elements, buttons, menus, and the overall functionality shown in detail."
    )
    
    # 3. Inisialisasi Converter
    print("Memuat model visi (SmolVLM) ke VRAM... Mohon tunggu.")
    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )
    
    # 4. Mulai Proses Ekstraksi & Inferensi
    print(f"Mulai memproses teks dan gambar dari {temp_pdf}...")
    result = converter.convert(temp_pdf)
    doc = result.document
    
    # (Opsional) Cek di terminal/log jika gambar berhasil dianotasi
    image_count = sum(1 for item, _ in doc.iterate_items() if isinstance(item, PictureItem))
    print(f"Selesai! Ditemukan {image_count} gambar/screenshot pada {max_pages} halaman ini.")
    
    # 5. SATUKAN SEMUA MENJADI TEKS UTUH
    # export_to_markdown() secara otomatis merangkai paragraf PDF dan anotasi VLM
    unified_text = doc.export_to_markdown()
    
    # Bersihkan file PDF sementara
    if os.path.exists(temp_pdf):
        os.remove(temp_pdf)
        
    return unified_text

# ==========================================
# CARA PENGGUNAAN
# ==========================================
pdf_file = "./data/MODUL PEMBELAJARAN.pdf" # Ganti dengan path file Anda

# Parse hanya 5 halaman pertama
hasil_teks_utuh = parse_pdf_to_unified_text(pdf_file, max_pages=5)

print("\n\n=== HASIL TEKS BERSATU (MARKDOWN) ===\n")
print(hasil_teks_utuh)

c:\Users\Lenovo\anaconda3\envs\law\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PDF berhasil dipotong menjadi 5 halaman pertama.
Memuat model visi (SmolVLM) ke VRAM... Mohon tunggu.
Mulai memproses teks dan gambar dari temp_subset.pdf...


[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 128002. This may result in unexpected behavior.
Loading weights: 100%|██████████| 471/471 [00:00<00:00, 2445.37it/s]
[INFO] 2026-08-11 00:01:38,508 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-11 00:01:38,523 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-11 00:01:38,551 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Lenovo\anaconda3\envs\law\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-11 00:01:38,552 [RapidOCR] main.py:50: Using C:\Users\Lenovo\anaconda3\envs\law\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-11 00:01:38,785 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-11 00:01:38,787 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-11 00:01:38,792 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Lenovo\anacond

KeyboardInterrupt: 

In [2]:
import os
# Pertahankan baris ini
os.environ["TORCH_COMPILE_DISABLE"] = "1"

from pypdf import PdfReader, PdfWriter
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions, 
    smolvlm_picture_description
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc import PictureItem

def parse_pdf_to_unified_text(pdf_path, max_pages=5):
    temp_pdf = "temp_subset.pdf"
    
    # (Fungsi prepare_pdf_subset asumsikan sudah ada di atas seperti sebelumnya)
    prepare_pdf_subset(pdf_path, temp_pdf, max_pages)
    
    # 2. Konfigurasi Pipeline Docling
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_picture_description = True  
    pipeline_options.picture_description_options = smolvlm_picture_description
    pipeline_options.picture_description_options.prompt = (
        "This is a screenshot of a web application interface. "
        "Describe the visible UI elements, buttons, menus, and the overall functionality shown in detail."
    )
    
    # ==========================================
    # SOLUSI UNTUK std::bad_alloc (OUT OF MEMORY)
    # ==========================================
    pipeline_options.num_threads = 1  # Wajib: Paksa proses 1 halaman bergantian
    # ==========================================
    
    print("Memuat model visi (SmolVLM) ke VRAM... Mohon tunggu.")
    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )
    
    print(f"Mulai memproses teks dan gambar dari {temp_pdf}...")
    result = converter.convert(temp_pdf)
    doc = result.document
    
    unified_text = doc.export_to_markdown()
    
    if os.path.exists(temp_pdf):
        os.remove(temp_pdf)
        
    return unified_text

# ==========================================
# CARA PENGGUNAAN
# ==========================================
pdf_file = "./data/MODUL PEMBELAJARAN.pdf"

# Saran tambahan: Coba ubah max_pages=1 atau 2 dulu untuk tes pertama
hasil_teks_utuh = parse_pdf_to_unified_text(pdf_file, max_pages=2) 

print("\n\n=== HASIL TEKS BERSATU (MARKDOWN) ===\n")
print(hasil_teks_utuh)

PDF berhasil dipotong menjadi 2 halaman pertama.


ValueError: "PdfPipelineOptions" object has no field "num_threads"

In [ ]:
import os
import gc  # Garbage Collector bawaan Python
# Wajib untuk menghindari error compiler C++
os.environ["TORCH_COMPILE_DISABLE"] = "1"

from pypdf import PdfReader, PdfWriter
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions, 
    smolvlm_picture_description
)
from docling.document_converter import DocumentConverter, PdfFormatOption

def parse_pdf_page_by_page(pdf_path, max_pages=5):
    # 1. Konfigurasi Docling (Tanpa num_threads yang error)
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_picture_description = True  
    pipeline_options.picture_description_options = smolvlm_picture_description
    pipeline_options.picture_description_options.prompt = (
        "This is a screenshot of a web application interface. "
        "Describe the visible UI elements, buttons, menus, and the overall functionality shown in detail."
    )
    
    print("Memuat model visi (SmolVLM) ke VRAM...")
    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )
    
    # 2. Baca PDF Asli
    reader = PdfReader(pdf_path)
    total_pages = min(max_pages, len(reader.pages))
    
    hasil_markdown_semua_halaman = []
    
    # 3. Proses LOOP: Satu Halaman Satu Waktu
    for i in range(total_pages):
        print(f"\n--- Memproses Halaman {i+1} dari {total_pages} ---")
        
        # Ekstrak Halaman ini ke PDF sementara
        temp_pdf = f"temp_page_{i}.pdf"
        writer = PdfWriter()
        writer.add_page(reader.pages[i])
        
        with open(temp_pdf, "wb") as f_out:
            writer.write(f_out)
            
        try:
            # Parse menggunakan Docling
            result = converter.convert(temp_pdf)
            doc = result.document
            
            # Ubah ke Markdown dan simpan ke list
            teks_halaman = doc.export_to_markdown()
            hasil_markdown_semua_halaman.append(teks_halaman)
            print(f"Sukses mengekstrak halaman {i+1}")
            
        except Exception as e:
            print(f"Gagal memproses halaman {i+1}. Error: {e}")
            
        finally:
            # Hapus PDF sementara
            if os.path.exists(temp_pdf):
                os.remove(temp_pdf)
                
            # ==================================================
            # PEMBERSIHAN MEMORI (Mencegah std::bad_alloc)
            # ==================================================
            gc.collect() # Bersihkan RAM CPU
            try:
                import torch
                if torch.cuda.is_available():
                    torch.cuda.empty_cache() # Bersihkan VRAM GPU
            except:
                pass

    # 4. Satukan semua teks Markdown dengan garis pemisah
    teks_utuh = "\n\n---\n\n".join(hasil_markdown_semua_halaman)
    return teks_utuh


# ==========================================
# CARA PENGGUNAAN
# ==========================================
pdf_file = "./data/MODUL PEMBELAJARAN.pdf"

# Mulai dengan max_pages=2 dulu untuk memastikan tidak ada error memori
hasil_teks_utuh = parse_pdf_page_by_page(pdf_file, max_pages=5)

print("\n\n=== HASIL TEKS BERSATU (MARKDOWN) ===\n")
print(hasil_teks_utuh)